# Plant networks: mDNS discovery, Sparkplug B and payload codecs

Find devices without a DNS server, give MQTT a state model with Sparkplug B, and read the binary payloads you meet on the way (Protobuf, MessagePack, TLV). Everything runs in this process.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.17.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.17.0-preview.1"
#r "nuget: IoTCom.Net.Serialization.Protobuf, 0.17.0-preview.1"
#r "nuget: IoTCom.Net.Serialization.MessagePack, 0.17.0-preview.1"

## Who is on the network?
A browser asks 224.0.0.251:5353 for every service type, then for the instances of each. `MdnsSimulator` runs a small plant segment in memory; `MdnsBrowser.Create()` without options uses the real network.

In [ ]:
using IoTCom.Net.Protocols.Mdns;

await using var plant = new MdnsSimulator();
await plant.StartAsync();
await using var browser = plant.Browser();
await browser.StartAsync();
foreach (var type in await browser.EnumerateTypesAsync(TimeSpan.FromMilliseconds(500)))
    foreach (var svc in await browser.BrowseAsync(type.Replace(".local", ""), TimeSpan.FromMilliseconds(300)))
        Console.WriteLine($"{svc.Type,-14} {svc.Instance,-24} {svc.Address}:{svc.Port}  {string.Join(" ", svc.Properties.Select(p => $"{p.Key}={p.Value}"))}");

## Sparkplug B: births, data and death
The edge node registers NDEATH as its MQTT will and publishes NBIRTH/DBIRTH with every metric and an alias. The host application resolves aliases in later DDATA messages.

In [ ]:
using System.Net;
using System.Net.Sockets;
using IoTCom.Net.Adapters.Mqtt;
using IoTCom.Net.Protocols.Sparkplug;

var probe = new TcpListener(IPAddress.Loopback, 0);
probe.Start();
var mqttPort = ((IPEndPoint)probe.LocalEndpoint).Port;
probe.Stop();
var broker = MqttBroker.Create(mqttPort);
await broker.StartAsync();
var scada = SparkplugHost.Create(o => { o.HostId = "notebook"; o.Mqtt = m => m.UseBroker("127.0.0.1", mqttPort); });
await scada.StartAsync();
var edge = SparkplugEdgeNode.Create(o => { o.Group = "Plant"; o.EdgeNode = "Line1"; o.Mqtt = m => m.UseBroker("127.0.0.1", mqttPort); });
var bottling = new SparkplugLineSimulator(edge).Define();
await edge.StartAsync();
for (var i = 0; i < 3; i++) { await bottling.Step(); await Task.Delay(200); }
foreach (var v in scada.Views.OrderBy(v => v.Key))
    Console.WriteLine($"{v.Key,-20} {(v.Online ? "online " : "offline")} {string.Join(", ", v.Metrics.Values.Where(m => m.Name != "Node Control/Rebirth").Select(m => m.ToString()))}");

## Commands and a pulled cable
Writes travel as DCMD and come back as DDATA. When the connection drops, the broker publishes the NDEATH will and the host marks the node and its devices offline.

In [ ]:
await scada.WriteAsync("Plant", "Line1", "Filler", "Running", false);
await Task.Delay(300);
Console.WriteLine($"Filler running: {scada.Find("Plant", "Line1", "Filler")!.Metrics["Running"].Value}");
await edge.DropConnectionAsync();
await Task.Delay(500);
Console.WriteLine($"Line1 online: {scada.Find("Plant", "Line1")!.Online} · next bdSeq {edge.BdSeq}");
await scada.DisposeAsync();
await broker.DisposeAsync();

## What a Sparkplug payload looks like
It is Protobuf: `SparkplugPayload.Describe` names the fields, `ProtobufWire` reads any message without a schema.

In [ ]:
using IoTCom.Net.Serialization.Protobuf;

var payload = new SparkplugPayload { Timestamp = 1, Seq = 4, Metrics = [SparkplugMetric.Of("Level", SparkplugDataType.Double, 61.5, alias: 7)] }.Encode();
Console.WriteLine(Convert.ToHexString(payload));
foreach (var f in SparkplugPayload.Describe(payload)) Console.WriteLine($"{f.Name,-10} {f.Value}");
ProtobufWire.TryInspect(payload, out var wire);
foreach (var f in wire) Console.WriteLine($"#{f.Number} wire {f.WireType}: {f.Value}");

## MessagePack and TLV
MessagePack is JSON-shaped binary; BER-TLV is the tag-length-value layout of smart cards and EMV.

In [ ]:
using IoTCom.Net.Serialization.MessagePack;
using IoTCom.Net.Serialization.Tlv;

var packed = new MessagePackCodec<Dictionary<string, double>>(MessagePack.MessagePackSerializerOptions.Standard).Encode(new() { ["temp"] = 27.5, ["rh"] = 71 });
Console.WriteLine($"{Convert.ToHexString(packed)} → {MessagePackView.ToJson(packed)}");
var fci = Convert.FromHexString("6F148407A0000000031010A5095004564953419F3800");
void Show(IEnumerable<TlvItem> items, string indent = "") { foreach (var i in items) { Console.WriteLine($"{indent}{i.Tag:X} {(i.Children.Count > 0 ? "" : Convert.ToHexString(i.Value))}"); Show(i.Children, indent + "  "); } }
Show(BerTlv.Decode(fci));

## Going further
`iotcom mdns browse`, `iotcom sparkplug watch --sim`, `iotcom payload protobuf <hex>`, the Gallery's *Plant network* demo and the MdnsDiscovery and SparkplugEdgeNode samples. See `docs/en/protocols/mdns.md`, `sparkplug.md` and `payload-codecs.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*